# 14 - Reading the ceiling

**Purpose.** To explain what notebook `13` established about `ceiling(gain)`, the full well and
the light bench, and what a reader should now believe about each. `13` is the notebook that
talked to the camera and made these numbers, and is written for someone *checking* the work.
This one is written for someone *deciding what to do next*.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `linearity_constants.json`, `linearity_rungs.csv`, `light_stability.csv`, and the
predecessors they lean on: `ptc_constants.json` for `g(gain)` and `bias_constants.json` for the
read noise. Where arithmetic appears below it is done on published numbers, to show what a
published number is worth; if any of it disagreed with `results/`, `results/` would be right and
this notebook would be the bug.

**It assumes `00_statistics.ipynb`** for why a plane mean over tens of thousands of pixels
resolves a hundredth of a count, and why a spread across repeats is the yardstick a departure has
to beat. It assumes `13` for the bench: the six gates, the balanced patch, and the rung grid.

**The headline: there is no bend, and that is the measurement.**

1. **`ceiling` is published null on all twelve (gain, plane) fits**, and the null carries a
   reason. Nowhere below the clip does the response depart 1% from its own line. The worst
   departure anywhere is 0.42 / 0.39 / 0.83% at gain 50 / 100 / 200, against reference lines
   whose own rungs scatter 0.23 / 0.23 / 0.41% about them. Twice the noise floor is enough to
   say *no bend* and nowhere near enough to say *this much curvature*.
2. **Three constants carry what the ladder did prove.** `linear_to_at_least` is the highest level
   shown straight (3958 / 3889 / 3854 counts), `worst_departure` is the quality of that proof,
   and `clip_level` is where the ladder actually stopped: the ADC's last code, at every gain.
3. **The converter binds, not the well.** Saturation moves 0.0002% across gains in counts and
   144% in electrons. A bend at a common level could be a well that happens to fill there; a
   straight line into the same top code at three gains spanning two octaves cannot be.
4. **L28's 3984-count bend is refuted, and section 3 shows what it most likely was.** The rungs
   that rule 6 throws out - the ones with pixels already pinned - cross 1% exactly where L28 put
   its bend, and their departure grows with the pinned fraction. That is clipping being read as
   non-linearity.
5. **The model consumes the bound, not the clip.** Section 5 prices that choice: it shortens the
   longest allowable sub by 3-6% and costs a few tenths of a percent of SNR, to stop the model
   spending a 3-6% region nobody measured.
6. **The bench is better than the retired one and is not finished.** L31's 1.79% is not
   reproduced - this bench does 0.089% at the same gain - but the panel does drift, at +0.314
   counts/min warm, and nobody has yet watched it from cold.

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
read = lambda n: json.loads((RESULTS / n).read_text())

K5 = read("linearity_constants.json")     # notebook 13, this session
K1 = read("bias_constants.json")          # session 01
K2 = read("ptc_constants.json")           # session 02

rungs = pd.read_csv(RESULTS / "linearity_rungs.csv")
stability = pd.read_csv(RESULTS / "light_stability.csv")

num = lambda d: {int(k): v for k, v in d.items()}
GAINS = sorted(num(K5["clip_level"]["value"]))
BOX = 256                                  # 13's analysis box; every table below is at it
FULL_SCALE = 4095
BEND_PCT, LINE_MAX_PCT = 1.0, 50.0         # rule 5's threshold, and the reference region

G = num(K2["system_gain"]["value"])                       # e- per ADC count
CLIP = num(K5["clip_level"]["value"])
BOUND = num(K5["linear_to_at_least"]["value"])
DEP = num(K5["worst_departure"]["value"])
RESID = num(K5["line_resid"]["value"])
WELL_CLIP = num(K5["full_well"]["value"])
WELL_BOUND = num(K5["full_well_at_linear_to"]["value"])
L28_BEND = 3984                            # the retired claim, a prediction and never an input

main = rungs[rungs.roi == BOX].copy()
PEDESTAL = main.groupby("gain").pedestal.mean().to_dict()

print("notebook 13 published %d constants on %s, from %d frames"
      % (len(K5), K5["clip_level"]["measured_on"], K5["clip_level"]["source_frames"]))
print("gains shot: %s;  gain 0 was dropped by gate 4 - %s"
      % (GAINS, list(K5["gains_out_of_reach"]["value"].values())[0]))
print("rung table: %d rows over %d planes and %d crop boxes"
      % (len(rungs), rungs.plane.nunique(), rungs.roi.nunique()))
print()
print("the twelve fits and their verdicts:")
print(pd.Series(K5["not_measured"]["value"]).to_string())

---

## 1. What was asked, and what the ladder answered

Rule 5 of `protocols/05-linearity.md` defines the ceiling as a **crossing**: fit a line through
the rungs below 50% of `t_sat`, force it through the origin in exposure, and find the level where
the measured response falls 1% below it. A crossing is only a measurement if there is one.

Two panels below. The left is the ladder itself - level against commanded exposure, one line per
gain - and it is straight by eye all the way into the flat top, which is the clip. That is the
whole finding, and everything after this section is about how confident one is allowed to be
about a straight line.

The right panel is the same data with the line divided out: the **departure**, in percent, of
each rung from the line its own low rungs define. Rule 5 is looking for the moment that curve
crosses -1%. The rungs rule 6 admits never get there. The rungs it excludes - open circles - do,
and section 3 is about why they do not count.

In [ ]:
# One (gain, plane): the reference slope, and each rung's departure from it.
# Rebuilt here exactly as 13 builds it -- low rungs only, forced through the
# origin -- so the curve plotted is the one rule 5 tested and not a second
# estimator that happens to agree with it.
def fitted(sub):
    sub = sub.sort_values("exptime")
    low = sub[(sub.rung_pct <= LINE_MAX_PCT) & sub.usable]
    k = float((low.signal * low.exptime).sum() / (low.exptime ** 2).sum())
    return k, sub.assign(dep=100 * (sub.signal / (k * sub.exptime) - 1.0))


curves = {key: fitted(sub)[1] for key, sub in main.groupby(["gain", "plane"])}

fig, ax = plt.subplots(1, 2, figsize=(10.0, 3.6))
colour = {50: "tab:blue", 100: "crimson", 200: "tab:green"}

for (g, p), c in curves.items():
    ax[0].plot(c.exptime, c.level, "-", lw=0.8, color=colour[g], alpha=0.75,
               label=f"gain {g}" if p == "G1" else None)
ax[0].axhline(FULL_SCALE, color="0.5", lw=0.8, ls=":", label="top code, 4095")
ax[0].set(xlabel="commanded exposure, s", ylabel="plane mean, ADC counts",
          title="the ladder: four planes, three gains")
ax[0].legend(fontsize=7)

for (g, p), c in curves.items():
    ok, bad = c[c.usable], c[~c.usable]
    ax[1].plot(ok.level, ok.dep, "o-", ms=3, lw=0.8, color=colour[g], alpha=0.8,
               label=f"gain {g}" if p == "G1" else None)
    ax[1].plot(bad.level, bad.dep, "o", ms=5, mfc="none", color=colour[g], alpha=0.8)
ax[1].axhline(-BEND_PCT, color="0.3", lw=1.0, ls="--", label="rule 5's 1% bend")
ax[1].axvline(L28_BEND, color="0.6", lw=0.9, ls=":", label="L28's claimed bend, 3984")
ax[1].set(xlabel="plane mean, ADC counts", ylabel="departure from own line, %",
          ylim=(-4.0, 1.0), title="open circles are the rungs rule 6 excludes")
ax[1].legend(fontsize=7)
plt.tight_layout()

print("last rung admitted by rule 6, and how far from straight the response got reaching it:")
tab = pd.DataFrame({"gain": GAINS,
                    "linear_to_at_least": [BOUND[g] for g in GAINS],
                    "as % of top code": [100 * BOUND[g] / FULL_SCALE for g in GAINS],
                    "worst departure %": [DEP[g] for g in GAINS],
                    "clip_level": [CLIP[g] for g in GAINS]})
print(tab.round(3).to_string(index=False))

## 2. Why "no bend" is a measurement and not a shrug

A null is worth exactly as much as the noise floor of the thing that failed to see anything. So
the departure has to be read against `line_resid`: how far the *reference* rungs - the ones
defining the line - scatter about it. That number is the resolution of this whole measurement.

| gain | worst departure | line scatter | ratio |
|---|---|---|---|
| 50 | 0.42% | 0.23% | 1.8x |
| 100 | 0.39% | 0.23% | 1.7x |
| 200 | 0.83% | 0.41% | 2.0x |

**Read that twice, because it says two things at once.** The departure is about twice the noise
floor, so it is not zero and this bench can see it. And it is a factor of two, not a factor of
ten, so the shape of it is not resolved: reporting "the sensor curves 0.42% before it clips"
would be reporting a number of which half could be the measurement's own scatter. `13` publishes
`worst_departure` next to `line_resid` for exactly this reason, and its note says so.

**What licenses the null is the size of the effect being looked for.** A 1% bend is a
well-separated target: it is four to five times the line scatter at every gain. If the sensor bent
1% below the clip, this ladder would have seen it clearly. It did not, at any gain, on any of four
planes, in any of four crop boxes - forty-eight independent chances to find one.

The third column below is the honest reason the null is safe: the departure would have to be more
than double what it is before rule 5 fires.

In [ ]:
resid_tab = pd.DataFrame({
    "gain": GAINS,
    "worst departure %": [DEP[g] for g in GAINS],
    "line scatter %": [RESID[g] for g in GAINS],
    "departure / scatter": [DEP[g] / RESID[g] for g in GAINS],
    "bend threshold / scatter": [BEND_PCT / RESID[g] for g in GAINS],
    "short of the threshold by": [BEND_PCT - DEP[g] for g in GAINS]})
print(resid_tab.round(3).to_string(index=False))
print()

# The repeat spread is the third view of the same floor, and it is independent of
# the fit: three frames at one rung, each already monitor-corrected, so what is
# left is the bench's own repeatability at that level.
usable = main[main.usable & (main.rung_pct > LINE_MAX_PCT)]
rep = usable.assign(rep_pct=100 * usable.repeat_spread / usable.signal).groupby("gain").rep_pct
print("repeat spread of three corrected frames at one rung, above %.0f%% of t_sat:"
      % LINE_MAX_PCT)
print(pd.DataFrame({"median %": rep.median(), "worst %": rep.max(),
                    "line scatter %": pd.Series(RESID)}).round(3).to_string())
print()
print("the three floors agree on the order: this bench resolves a few tenths of a percent,")
print("and rule 5 was asked to find one percent.  That gap is what makes the null publishable.")

fig, ax = plt.subplots(figsize=(6.0, 3.2))
x = np.arange(len(GAINS))
ax.bar(x - 0.18, [DEP[g] for g in GAINS], width=0.36, color="crimson",
       label="worst departure reached")
ax.bar(x + 0.18, [RESID[g] for g in GAINS], width=0.36, color="0.55",
       label="scatter of the reference line itself")
ax.axhline(BEND_PCT, color="0.2", lw=1.2, ls="--", label="rule 5's bend threshold")
ax.set(xticks=x, xticklabels=[f"gain {g}" for g in GAINS], ylabel="percent",
       title="what was found, against what could be resolved, against what was sought")
ax.legend(fontsize=7)
plt.tight_layout()

## 3. What L28 most likely measured: the onset of clipping

L28 claimed a 1% bend at **3984 counts, 97.3% of the top code, measured twice to 0.05%**. This
ladder put rungs either side of 3984 and the response through them is straight to 0.42%. So the
claim is refuted - but a number reproduced twice to 0.05% is not noise, and the interesting
question is what it *was*.

**The answer is in the rungs rule 6 throws away.** A plane mean is dragged down by every pixel
sitting at the top code, because a clipped pixel cannot contribute the value it would have had.
That drag is not the sensor lying; it is the mean no longer being the mean. Rule 6 exists to
exclude those rungs, and it was written before the session rather than after it.

Plot the excluded rungs' departure against their pinned fraction and the mechanism is plain: the
departure is a **function of how many pixels are pinned**, not of the level. Rungs with under a
percent pinned sit at a few tenths of a percent; rungs a third pinned cross -1%; rungs almost
wholly pinned reach -5%.

**And the crossing lands where L28 put its bend.** The lowest excluded rung past -1% sits at
4047 counts, one rung above L28's 3984 - and that is from a bench that is not this one, with its
own illumination map and its own hot pixels deciding which rung pins first. The
reading that fits all the evidence is that L28 measured the onset of clipping and called it
non-linearity. It is reproducible to 0.05% because clipping *is* reproducible: it is the same top
code every time.

In [ ]:
excluded = pd.concat(
    [c[(c.pinned_frac > 1e-4) & (c.pinned_frac < 0.999)].assign(gain=g, plane=p)
     for (g, p), c in curves.items()])

fig, ax = plt.subplots(1, 2, figsize=(10.0, 3.4))
for g in GAINS:
    s = excluded[excluded.gain == g]
    ax[0].plot(100 * s.pinned_frac, s.dep, "o", ms=6, color=colour[g], label=f"gain {g}")
ax[0].axhline(-BEND_PCT, color="0.3", lw=1.0, ls="--", label="rule 5's 1% bend")
ax[0].set(xscale="log", xlabel="pixels already at the top code, % of plane",
          ylabel="departure from own line, %",
          title="the departure tracks the pinning, not the level")
ax[0].legend(fontsize=7)

for (g, p), c in curves.items():
    ok = c[c.usable]
    ax[1].plot(ok.level, ok.dep, "-", lw=0.7, color=colour[g], alpha=0.6)
s = excluded
ax[1].plot(s.level, s.dep, "o", ms=5, mfc="none", color="0.2",
           label="rungs rule 6 excludes")
ax[1].axhline(-BEND_PCT, color="0.3", lw=1.0, ls="--")
ax[1].axvline(L28_BEND, color="crimson", lw=1.0, ls=":", label="L28's 3984")
ax[1].set(xlim=(3600, 4120), ylim=(-4.0, 0.6), xlabel="plane mean, ADC counts",
          ylabel="departure, %", title="where a fit that kept them would put a bend")
ax[1].legend(fontsize=7)
plt.tight_layout()

crossed = excluded[excluded.dep <= -BEND_PCT]
print("excluded rungs that do cross -1%%: %d of %d" % (len(crossed), len(excluded)))
print("  their levels run %.0f to %.0f counts, %.1f%% to %.1f%% of the top code"
      % (crossed.level.min(), crossed.level.max(),
         100 * crossed.level.min() / FULL_SCALE, 100 * crossed.level.max() / FULL_SCALE))
print("  their pinned fractions run %.1f%% to %.1f%%"
      % (100 * crossed.pinned_frac.min(), 100 * crossed.pinned_frac.max()))
print("  L28's claim: %d counts, %.1f%% of the top code"
      % (L28_BEND, 100 * L28_BEND / FULL_SCALE))
print()
print("admitted rungs, for contrast - none of them crosses:")
adm = pd.concat([c[c.usable & (c.rung_pct > LINE_MAX_PCT)] for c in curves.values()])
print("  %d rungs, worst departure %.3f%%, deepest level %.0f counts"
      % (len(adm), adm.dep.min(), adm.level.max()))

## 4. Which of the three stops binds, and why the answer got stronger

MISSION names three things that can end a pixel's useful range: the ADC runs out of codes, the
well fills, or the response bends first. Rule 8 was written to tell them apart using the bend
level across gains - a level fixed in **counts** means the converter, a level fixed in
**electrons** means the well.

With no bend, rule 8 runs on `clip_level` instead, and the logic is stronger that way rather than
weaker. Saturation is the same code at three gains spanning two octaves, to **0.0002%**, while
the charge that code represents falls as `1/g` - from 21718 e- at gain 50 to 3710 e- at gain 200,
a spread of **144%**.

**Why that is the stronger form.** A bend at a common level is *consistent* with a well that
happens to fill near there; you would need more gains to separate the two. A straight response
running into the same top code at every gain is not consistent with a well at all: if the well
were binding, the charge at saturation would be constant and the count would move.

The practical consequence is the one the model cares about: **full well is not a property of this
sensor, it is a property of the gain you chose.** Every gain step spends well.

In [ ]:
well = pd.DataFrame({
    "gain": GAINS,
    "g, e-/count": [G[g] for g in GAINS],
    "clip_level, counts": [CLIP[g] for g in GAINS],
    "pedestal, counts": [PEDESTAL[g] for g in GAINS],
    "full_well, e-": [WELL_CLIP[g] for g in GAINS],
    "full_well_at_linear_to, e-": [WELL_BOUND[g] for g in GAINS]})
print(well.round(3).to_string(index=False))
print()
spread = lambda v: 100 * (max(v) - min(v)) / np.mean(v)
print("across the three gains, saturation varies:")
print("  in ADC counts    %.4f%%   <- flat, so this is what binds" % spread(list(CLIP.values())))
print("  in electrons   %.2f%%" % spread(list(WELL_CLIP.values())))
print("verdict published by 13: bend_follows = %r" % K5["bend_follows"]["value"])

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.2))
ax[0].plot(GAINS, [CLIP[g] for g in GAINS], "o-", ms=7, color="tab:blue")
ax[0].set(ylim=(4094.5, 4095.5), xlabel="gain", ylabel="saturation, ADC counts",
          title="in counts: the same code, to 0.0002%")
ax[1].plot(GAINS, [WELL_CLIP[g] for g in GAINS], "o-", ms=7, color="crimson",
           label="at the clip")
ax[1].plot(GAINS, [WELL_BOUND[g] for g in GAINS], "s--", ms=6, color="0.4",
           label="at linear_to_at_least")
ax[1].set(yscale="log", xlabel="gain", ylabel="full well, e-",
          title="in electrons: it falls as 1/g")
ax[1].legend(fontsize=7)
plt.tight_layout()

## 5. What the model consumes, and what the safe choice costs

`ceiling(gain)` in MISSION's star-colour constraint is now **`linear_to_at_least`**, the highest
level this ladder proved straight - not `clip_level`, which is the truer physical statement about
the sensor. The two differ by 3-6%.

**The asymmetry decides it.** Under-exposing a star by a few percent costs a little SNR on that
star. Over-running into counts nobody proved straight costs a star colour, which is the quantity
the constraint exists to protect. The 3-6% between the bound and the clip is not a region shown
to be good - it is a region this ladder has nothing to say about, because the pixels there are
pinned.

**Here is the price, in the model's own terms.** The constraint says a star's peak may reach the
ceiling and no further, so the longest allowable sub scales with `(ceiling - pedestal)`. Taking
the bound instead of the clip shortens that sub by the same 3-6%. At fixed total time, the SNR of
faint extended signal depends on `t` only through `sqrt(t / (t + t_dead))` and `R^2/t`, and both
are weak there - so a 3-6% cut in `t` costs **0.7% of the SNR at a 60 s sub and 0.1-0.2% at
300 s**. It shrinks as the sub lengthens, because both weak terms do.

The sky rate below is L32's inherited figure and the dead time is a placeholder - neither is a
measured constant of this project yet, and both are labelled in the output. They set the scale of
the answer, not its sign: the penalty is small for any plausible pair, which is the whole point.

In [ ]:
SKY_E_PER_S = 1.594        # L32, green, unfiltered, Bortle 5-6 - inherited, not ours yet
T_DEAD_S = 10.0            # placeholder: MISSION lists t_dead as still to be measured
R_COUNTS = K1["read_noise_at_hcg"]["value"]

cost = []
for g in GAINS:
    head_clip = CLIP[g] - PEDESTAL[g]
    head_bound = BOUND[g] - PEDESTAL[g]
    ratio = head_bound / head_clip
    R_e = R_COUNTS * G[g]                 # read noise in electrons at this gain
    row = {"gain": g, "headroom at clip": head_clip, "headroom at bound": head_bound,
           "sub shortened by %": 100 * (1 - ratio), "R, e-": R_e}
    for t in (60.0, 120.0, 300.0):
        snr = lambda tt: (np.sqrt(tt / (tt + T_DEAD_S))
                          / np.sqrt(SKY_E_PER_S + R_e ** 2 / tt))
        row[f"SNR cost at t={t:.0f}s, %"] = 100 * (1 - snr(ratio * t) / snr(t))
    cost.append(row)

print("read noise %.4f counts at the HCG boundary (session 01), in electrons per gain above"
      % R_COUNTS)
print("sky %.3f e-/px/s is L32's inherited figure; t_dead %.0f s is a placeholder"
      % (SKY_E_PER_S, T_DEAD_S))
print()
print(pd.DataFrame(cost).round(3).to_string(index=False))
print()
worst_snr = max(r[k] for r in cost for k in r if k.startswith("SNR cost"))
print("the trade, stated plainly: give up %.1f-%.1f%% of sub length, and with it at worst %.2f%%"
      % (min(r["sub shortened by %"] for r in cost),
         max(r["sub shortened by %"] for r in cost), worst_snr))
print("of the SNR, to stop the model spending a region nobody measured.")
print()
print("what the same choice is worth in electrons, which is how the well is quoted:")
print(pd.DataFrame({"gain": GAINS,
                    "full_well (clip), e-": [WELL_CLIP[g] for g in GAINS],
                    "full_well_at_linear_to, e-": [WELL_BOUND[g] for g in GAINS],
                    "given up, e-": [WELL_CLIP[g] - WELL_BOUND[g] for g in GAINS]}
                   ).round(1).to_string(index=False))

## 6. The bench itself, and the one question left open about it

Three of this session's constants are about the *light*, not the camera, and they are what makes
the ladder above believable.

**Gate 4 balanced the source so four planes could be measured at once.** One white-ish panel
gives four different fluxes through the Bayer filters, and a ladder scaled to the brightest plane
runs the dimmest over a fraction of its own range - three of the four measurements simply cannot
be made. Solving for a patch colour per gain got the four planes inside 3.3-4.6% of each other,
against a 5% bar. The residual imbalance is not a nuisance left over; it is **the whole of the
per-plane scatter** quoted on `linear_to_at_least`. A plane running 2% bright pins one rung
earlier and its bound lands one rung lower, which is why that scatter shrinks with gain exactly
as the balance does.

**Gate 5 arm 1 found a real drift: +0.314 +/- 0.047 counts per minute, panel warm.** Seven sigma
from zero, and upstream of the sensor - session 01's dark arm over the same timescale was
-0.00133 +/- 0.254 counts/min with the light taken out. The per-rung monitor bracket is what
absorbs it, and `monitor_factor` shows it doing so.

**Gate 5 arm 2 is the arm the retired project could not run**, because its two gains differed in
exposure length and in elapsed time at once. Interleaving a short and a long exposure at one gain
separates them: the long/short flux ratio is 0.9886, with repeat scatter of 0.089% long and
0.299% short against L31's 1.79% at the same gain. **L31's irreproducibility is not reproduced**
- this bench is six times better - and the redraw worry dies with it, since light arriving in
pulses can only distort a ladder if the pulse count is not proportional to the shutter time.

**What is left of L31 is one untested mechanism: backlight thermal drift from cold.** Arm 1 ran
warm, which is the one state that cannot tell a panel still heating from a panel that has settled.
So `light-source.md`'s ten-minute warm-up is still a ritual whose reason is neither confirmed nor
falsified, and the entry stays in the queue narrowed to that.

In [ ]:
drift = stability[stability.arm == "drift"]
alt = stability[stability.arm != "drift"].copy()

fig, ax = plt.subplots(1, 2, figsize=(10.0, 3.2))
slope, inter = np.polyfit(drift.minutes, drift.level, 1)
ax[0].plot(drift.minutes, drift.level, "o", ms=4, color="0.35")
ax[0].plot(drift.minutes, inter + slope * drift.minutes, "-", lw=1.2, color="crimson",
           label="%+.3f counts/min" % slope)
ax[0].set(xlabel="minutes from the first frame", ylabel="plane mean, ADC counts",
          title="arm 1: the panel drifts, warm")
ax[0].legend(fontsize=7)

for name, s in alt.groupby(alt.exptime > alt.exptime.median()):
    label = "long" if name else "short"
    ax[1].plot(s.minutes, s.flux / alt[alt.exptime > alt.exptime.median()].flux.mean(),
               "o-", ms=4, lw=0.8, label="%s, %.1f s" % (label, s.exptime.mean()))
ax[1].set(xlabel="minutes", ylabel="counts per second, normalised to the long arm",
          title="arm 2: flux does not care how long the shutter was open")
ax[1].legend(fontsize=7)
plt.tight_layout()

mf = main.groupby("gain").monitor_factor
print("what the monitor bracket had to absorb, per gain (1.0 is a panel that did not move):")
print(pd.DataFrame({"min": mf.min(), "max": mf.max(),
                    "span %": 100 * (mf.max() - mf.min())}).round(4).to_string())
print()
print("balance achieved per gain, against gate 4's 5% bar, and the bound scatter it explains:")
print(pd.DataFrame({
    "gain": GAINS,
    "plane balance %": [num(K5["plane_balance"]["value"])[g] for g in GAINS],
    "bound scatter over planes": [num(K5["linear_to_at_least"]["uncertainty"])[g]
                                  for g in GAINS],
    "t_sat, s": [num(K5["t_sat_per_gain"]["value"])[g] for g in GAINS],
    "patch RGB": [num(K5["patch_colour_per_gain"]["value"])[g] for g in GAINS],
}).round(3).to_string(index=False))
print()
print("panel redraw period %.1f ms; the shortest rung on disk is %.0f redraws long,"
      % (K5["panel_redraw_period"]["value"], main.periods.min()))
print("so one pulse in N is %.2f%% against the %.0f%% a bend would have to be."
      % (100 / main.periods.min(), BEND_PCT))

## 7. L09's illumination map is real; its ROI rule is not load-bearing

L09 said: use a small ROI for linearity, because uneven illumination smears the bend over more
range than the bend itself. Gate 6 measured the illumination directly and **reproduced L09's
ordering** - 0.20% spread at the central 128 box rising to 3.36% at 1024, against L09's 0.53% and
3.8%. The map is real, and this bench is about half a percent flatter than theirs.

**The consequence L09 predicted does not appear.** Across those four boxes the bound moves 0.30%
and in the *opposite* direction to the prediction that a wide box reads low, and the curvature -
the continuous quantity, and therefore the sharper test - moves 0.014 percentage points. Both are
far inside the effect they would need to have to matter.

So the central 256 box is kept because it costs nothing, **not** because a wider one was shown to
fail. That distinction is the whole reason `roi_curvature` is published alongside
`roi_sensitivity`: the bound is quantised to one rung and can only show a gross effect, while the
curvature is continuous and shows none.

In [ ]:
roi = pd.DataFrame({
    "box": sorted(num(K5["roi_sensitivity"]["value"])),
    "bound moves %": [num(K5["roi_sensitivity"]["value"])[b]
                      for b in sorted(num(K5["roi_sensitivity"]["value"]))],
    "worst departure %": [num(K5["roi_curvature"]["value"])[b]
                          for b in sorted(num(K5["roi_curvature"]["value"]))]})
print(roi.round(4).to_string(index=False))
print()
print("L09 predicted the illumination spread as 0.53% at 256 and 3.8% at 1024, and claimed a")
print("wide box reads the level low.  The spread reproduced; the consequence did not:")
print("  bound moves %.3f%% over an eightfold change in box area per side, wrong sign"
      % (roi["bound moves %"].max() - roi["bound moves %"].min()))
print("  curvature moves %.3f percentage points, which is nothing"
      % (roi["worst departure %"].max() - roi["worst departure %"].min()))

fig, ax = plt.subplots(figsize=(5.8, 3.0))
ax.plot(roi.box, roi["worst departure %"], "o-", ms=7, color="crimson",
        label="worst departure from straight")
ax.set(xscale="log", xticks=roi.box, xticklabels=roi.box, xlabel="analysis box, mosaic px",
       ylim=(0, 1.1), ylabel="percent",
       title="L09's mechanism predicts this rising; it is flat")
ax.axhline(BEND_PCT, color="0.3", lw=1.0, ls="--", label="rule 5's bend threshold")
ax.legend(fontsize=7)
plt.tight_layout()

## 8. What is settled, and what the next session inherits

**Settled, and available to every later notebook:**

| constant | value | what it means |
|---|---|---|
| `ceiling`, `ceiling_per_plane` | null, twelve times | no 1% departure exists below the clip; `not_measured` says so per fit |
| `linear_to_at_least` | 3958 / 3889 / 3854 counts | the highest level proved straight - **the model's `ceiling(gain)`** |
| `worst_departure` | 0.42 / 0.39 / 0.83% | how far from straight it actually got, and the quality of that bound |
| `line_resid` | 0.23 / 0.23 / 0.41% | the noise floor that makes the line above readable |
| `clip_level` | 4095.000 / 4094.997 / 4094.994 | the ADC's last code, arrived at by a straight response |
| `full_well` | 21718 / 12027 / 3710 e- | what a pixel holds when the codes run out |
| `full_well_at_linear_to` | 20982 / 11413 / 3488 e- | the conservative well, which is what the model consumes |
| `bend_follows` | `converter` | at every gain shot, and by a stronger argument than rule 8 was designed to make |
| `roi_sensitivity`, `roi_curvature` | 0.30%, 0.014 pp | L09's map is real, its ROI rule is not load-bearing here |
| `light_drift`, `flux_vs_exposure_length` | +0.314 counts/min, 0.9886 | the bench drifts slowly and does not care about shutter length |

**Nothing here changes a capture setting on its own.** What it changes is a bound: the model may
now put a number on the star-colour constraint that came from this rig rather than from a spec
sheet or an inherited claim, and section 5 prices the safety margin at about a tenth of a percent
of SNR.

**Three gaps, named honestly.**

- **Gain 0 was never shot.** Gate 4 could not balance the four planes better than 8.3% against a
  5% bar, so it is published as out of reach rather than estimated. The constraint at gain 0 is
  therefore unmeasured, and `g(gain)` being largest there is exactly where the well is biggest
  and the constraint loosest - so this is the least costly gain to be missing, but it is missing.
- **Gain 450 was never shot either**, and the gain domain runs to it. The three gains here span
  two octaves and agree to 0.0002% on the thing that binds, which is a strong argument that the
  fourth would too - but it is an argument, not a measurement.
- **The 3-6% above the bound is unmeasured, by construction.** Nothing bends there; nothing was
  shown not to, either. A ladder with finer rungs near the top would narrow it, and would cost a
  bench night for a few percent of exposure headroom.

**What the next build step inherits.** Every camera-side constant the model consumes is now
measured. What is left is on-sky: `F_sky` per CFA plane, `t_dead` from frame timestamps, and
`eta_comb` against real stacks - and L32's inherited sky rate, used as a placeholder in section 5,
is the first of those to check against our own frames. L31 goes forward narrowed to a single
question about the bench light, which costs one cold-start block and no ladder at all.